In [ ]:
nvidia-smi -l 10  
# 每隔10秒刷新一次cpu状态

In [ ]:
# 1. 创建核心目录
export PROJECT_PATH=/home/aistudio/models/realityscan/GardenDeskOriginal
mkdir -p $PROJECT_PATH/images
mkdir -p $PROJECT_PATH/sparse

# 2. 检查路径
echo "请确保你的 100 张照片已存放在: $PROJECT_PATH/input/images"
# db文件存在根目录，images存放原始图片，sparse输出重建结果

In [ ]:
#gpu version of colmap
# 特征提取

# need to set use_gpu 0  for cpu,cannot use gpu for some reason
colmap feature_extractor \
    --database_path $PROJECT_PATH/database.db \
    --image_path $PROJECT_PATH/images \
    --ImageReader.single_camera 0 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 1 \
    --SiftExtraction.num_threads 64

# 穷举匹配 (100张图大约需要 100 分钟)  gpu only 1 minutes
colmap exhaustive_matcher \
    --database_path $PROJECT_PATH/database.db \
    --SiftMatching.use_gpu 1 \
    --SiftMatching.num_threads 64


#前两部的结果就是写入database.db中，可以直接用这个数据库重建

# 稀疏重建
# gpu for 8 minutes,cpu for 2hours
colmap mapper \
    --database_path $PROJECT_PATH/database.db \
    --image_path $PROJECT_PATH/images \
    --output_path $PROJECT_PATH/sparse \
    --Mapper.num_threads 64 \
    

In [ ]:
mkdir -p $PROJECT_PATH/undistorted
colmap image_undistorter \
    --image_path $PROJECT_PATH/images \
    --input_path $PROJECT_PATH/sparse/0 \
    --output_path $PROJECT_PATH/undistorted \
    --output_type COLMAP

# sparse/0 is the colmap output  dense/undistorted is undistorter output,will be used by gaussian splatting 
# undistorter  will put camera.bin point3D.bin to dense/undistorted/sparse  but we need it to be  dense/undistorted/sparse/0 
# make */0 folder move *.bin to the folder

mkdir -p $PROJECT_PATH/undistorted/sparse/0
mv $PROJECT_PATH/undistorted/sparse/*.bin $PROJECT_PATH/undistorted/sparse/0

In [ ]:

python /home/aistudio/models/gaussian-splatting/train.py \
    -s $PROJECT_PATH/undistorted \
    --model_path $PROJECT_PATH/output \
    --save_iterations 7000 21000 30000 \
    --iterations 30000

In [ ]:
cd models 
git clone https://github.com/graphdeco-inria/gaussian-splatting.git
cd gaussian-splatting
git submodule update --init --recursive --depth 1
cd gaussian-splatting
sorce activate MyEnvForColmap
pip install /home/aistudio/models/gaussian-splatting/submodules/simple-knn --no-build-isolation
pip install /home/aistudio/models/gaussian-splatting/submodules/diff-gaussian-rasterization --no-build-isolation